# CMA-StrongSORT — reproducing the paper (MOT17, MOT20, DanceTrack)

One notebook for every step. Set the options in the **Settings** cell and run all cells.

| LANGKAH | What it does | Runtime |
|---|---|---|
| `all` | downloads the datasets once, trains the detectors (YOLOv8m, GSW-YOLO) and the ReID models, detects, embeds and tracks — every unit that is not finished yet | GPU (T4 or A100) |
| `status` | shows which units are done, running or waiting | CPU |
| `eval` | TrackEval on all tracker outputs and the tables of the paper (`results/tables.md`, `tables.tex`) | CPU |
| `speed` | timing table (detect / embed / CMC / association, FPS, peak GPU memory) | the GPU of the paper (T4) |

Several sessions may run `all` at the same time (second one with `REVERSE = True`). Units are locked; an interrupted
unit continues in the next session (detector and ReID training resume from their last epoch).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ====================== SETTINGS ======================
WORK = '/content/drive/MyDrive/CMA_StrongSORT'          # results, models, dataset archives (about 35 GB)
CODE = 'https://github.com/januaradip/CMA-StrongSORT.git' # or the path of CMA-StrongSORT.zip on Drive
LANGKAH = 'all'            # all / status / eval / speed
REVERSE = False            # True in a second parallel session
DATASETS = []              # empty = MOT17, MOT20, DanceTrack; e.g. ['MOT17']
DETECTORS = []             # empty = YOLOv8, GSW-YOLO
EXTRA = []                 # pipeline.CFG overrides, e.g. ['DET_EPOCHS=100']
LOCAL = '/content/data'    # extracted datasets (fast local disk)

In [ ]:
import os, glob, sys, subprocess
if CODE.endswith('.zip'):
    !rm -rf /content/CMA_code && mkdir -p /content/CMA_code && unzip -q -o "{CODE}" -d /content/CMA_code
    REPO = os.path.dirname(glob.glob('/content/CMA_code/**/run.py', recursive=True)[0])
else:
    !rm -rf /content/CMA-StrongSORT && git clone -q {CODE} /content/CMA-StrongSORT
    REPO = '/content/CMA-StrongSORT'
!pip install -q -r {REPO}/requirements.txt
os.makedirs(f'{WORK}/results', exist_ok=True)
env = subprocess.run([sys.executable, '-c', 'import sys, torch, ultralytics, boxmot; print("python", sys.version.split()[0]); '
                      'print("torch", torch.__version__, "cuda", torch.version.cuda); print("ultralytics", ultralytics.__version__); '
                      'print("boxmot", boxmot.__version__); print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")'],
                     capture_output=True, text=True).stdout
print(env); open(f'{WORK}/results/environment.txt', 'a').write(env + '\n')
%cd {REPO}

In [ ]:
args = f'--work "{WORK}" --local "{LOCAL}" --step {LANGKAH}'
if DATASETS:  args += ' --datasets ' + ' '.join(DATASETS)
if DETECTORS: args += ' --detectors ' + ' '.join(DETECTORS)
if REVERSE:   args += ' --reverse'
if EXTRA:     args += ' --set ' + ' '.join(EXTRA)
!python run.py {args}

### Progress and stale locks
A unit stays locked while its session works on it. If a session died, its lock becomes stale after 3 hours; the
cell below removes stale locks immediately (only run it when no other session is working on that unit).

In [ ]:
!python run.py --work "{WORK}" --step status
import time
for lp in glob.glob(f'{WORK}/**/LOCK', recursive=True):
    age = (time.time() - os.path.getmtime(lp)) / 60
    print(f'{age:6.0f} min  {os.path.relpath(lp, WORK)}')
# to remove a lock of a session that is dead:   os.remove(f'{WORK}/<path shown above>')

### Send the results
Zips the tables, metrics and run information (no images or weights) to `hasil_cma_strongsort.zip` on Drive.

In [ ]:
import shutil, zipfile
out = f'{WORK}/hasil_cma_strongsort.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{WORK}/results/**/*', recursive=True) + glob.glob(f'{WORK}/**/DONE.json', recursive=True) \
             + glob.glob(f'{WORK}/tracks/**/stats.json', recursive=True) + glob.glob(f'{WORK}/detectors/*/train/results.csv'):
        if os.path.isfile(p):
            z.write(p, os.path.relpath(p, WORK))
print(out, round(os.path.getsize(out) / 1e6, 2), 'MB')